# F1: End-to-end audit trail
Each guarded call writes one MLflow trace (experiment `aigov-audit`, workspace `ai-governance-demo`): request ID, end user, app, API key,
subscription, model and version, time, **masked** prompt and response, guardrail decision. Auditors get read-only MLflow access only.
Per-request gateway records (subscription, user, model, status) are also in Loki (`usage` LokiStack, TP).

In [ ]:
%pip install -q "mlflow[kubernetes]==3.14.0" 

The install message "you may need to restart the kernel" can be ignored the first time; restart the kernel only if the next cells fail to import `mlflow`.

In [ ]:
import os, sys, json
os.chdir(os.path.expanduser("~/aigov"))           # notebooks/ and data/ live here
sys.path.insert(0, "notebooks")
import harness as h
print("Guardrails:", os.environ["NEMO_URL"])

In [ ]:
import audit_client as a
for user, cfg, prompt in a.DEMO:
    r = a.audited_call(prompt, user, cfg)
    print(f'{r["request_id"]}  {r["decision"]:15s} {",".join(r["pii"]) or "-":28s} {r["audit_text"][:70]}')

**Reading the output:** one line per demo request, as the application's audit client records it:

* first column: the **request id**, which is also the trace id in MLflow, so any request can be looked up.
* decision: `masked` (PII replaced, then answered), `blocked_input` (stopped before the model, for example an injection attempt), `blocked_output` (answer stopped by the output check), `allowed` (no PII, answered normally).
* PII types found, for example `NIK,PHONE` (`-` = none).
* the **masked** text that is stored: placeholders instead of values, or `<blocked before model>`. No raw personal data is written to the audit trail.

In [ ]:
from mlflow import MlflowClient
c = MlflowClient(); e = c.get_experiment_by_name("aigov-audit")
t = c.search_traces(locations=[e.experiment_id], max_results=1)[0]
{k: t.info.tags.get(k) for k in ("request_id","end_user","app","api_key_name","subscription","model_id","model_uri","guardrail_decision","rails_triggered","pii_types")}

**Reading the output:** the tags stored on the most recent trace, which an auditor sees in MLflow:

| Tag | Meaning |
|---|---|
| `request_id` | unique id of the request (same as the trace id) |
| `end_user` | the customer or batch the request was made for |
| `app` | which application made the call |
| `api_key_name` | which MaaS API key was used |
| `subscription` | the team subscription billed for it |
| `model_id` / `model_uri` | the model and its exact image version |
| `guardrail_decision` | masked / blocked_input / blocked_output / allowed |
| `rails_triggered` | which guardrail rules fired |
| `pii_types` | the kinds of personal data found (types only, never values) |